In [19]:
import numpy as np
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister
from qiskit.circuit.library import ModularAdderGate

# DeepSeek code begins
def sample_path(n, k):
    """
    Classically sample a random path on the graph G_k (vertices 1..k-1)
    according to the distribution Pr(p) ∝ λ_{\ell(p)}.

    Args:
        n (int): length of the path (number of strands)
        k (int): parameter of the Jones polynomial (k-th root of unity)

    Returns:
        path (list[int]): list of n bits (0/1) representing the steps
                          (0 = left, 1 = right) starting from vertex 1.
        ell (int): final vertex (1..k-1) that the sampled path ends at.
    """
    # S[i][j] = number of valid i‑step paths from vertex 1 to vertex j
    # We use indices 0..k (0 and k are always zero for convenience)
    S = np.zeros((n + 1, k + 1), dtype=np.float64)
    S[0, 1] = 1.0                     # length 0: only at start vertex 1
    
    for i in range(1, n + 1):
        for j in range(1, k):
            left  = S[i-1, j-1] if j-1 >= 1 else 0.0
            right = S[i-1, j+1] if j+1 <= k-1 else 0.0
            S[i, j] = left + right

    # ── 2. Choose final vertex ℓ with probability ∝ λ_ℓ * S[n, ℓ] ──
    weights = [lam[ell] * S[n, ell] for ell in range(1, k)]
    total   = sum(weights)
    # pick a random vertex using the cumulative distribution
    r = np.random.random() * total
    cumulative = 0.0
    ell = 1
    for idx, w in enumerate(weights, start=1):
        cumulative += w
        if r <= cumulative:
            ell = idx
            break

    # ── 3. Backward sampling of the whole path ──
    path = [0] * n        # will store the forward step bits (1 = right, 0 = left)
    v = ell               # current vertex (we go backwards from step n to 1)

    for i in range(n, 0, -1):
        # possible previous vertices
        prev_left  = v - 1 if v - 1 >= 1 else None
        prev_right = v + 1 if v + 1 <= k - 1 else None

        if prev_left is None:            # only right move possible
            predecessor = prev_right
            bit = 0                      # coming from v+1 → forward was left (0)
        elif prev_right is None:         # only left move possible
            predecessor = prev_left
            bit = 1                      # coming from v-1 → forward was right (1)
        else:
            # both possible: choose with probability ∝ S[i-1][prev]
            prob_left = S[i-1, prev_left] / (S[i-1, prev_left] + S[i-1, prev_right])
            if np.random.random() < prob_left:
                predecessor = prev_left
                bit = 1                  # right move in forward direction
            else:
                predecessor = prev_right
                bit = 0                  # left move in forward direction

        path[i-1] = bit
        v = predecessor

    return path
# DeepSeek code ends

def create_braidGate(strand_count, braid_sequence, k):
    '''
    A function that returns the gate that executes the braid representation of braid_sequence in B_(strand_count).
    k is a parameter relevant to approximating the Jones polynomial of said braid at k-th root of unity.
    braid_sequence is a list of nonzero integers. Negative integers denote inverse of the respective generator. List is read left to right.
    Reference: "Aharonov, et al - A Polynomial Quantum Algorithm for Approximating the Jones Polynomial"
    '''
    #Creating the quantum circuit that will be turned into a gate
    path = QuantumRegister(strand_count,'path')
    anc = QuantumRegister(anc_size, 'anc')
    const = QuantumRegister(anc_size,'const')
    circ = QuantumCircuit(path, anc, const)

    #Creating a gate that "walks" and "unwalks" the ancilla
    adder = ModularAdderGate(anc_size)
    ctrl_adder = adder.control(1)

    ctrl_walk = QuantumRegister(1,'ctrl')
    anc_walk = QuantumRegister(anc_size, 'anc')
    const_walk = QuantumRegister(anc_size,'const')
    walk = QuantumCircuit(ctrl_walk, anc_walk, const_walk)

    walk.x(const_walk[0])
    walk.append(ctrl_adder, [ctrl_walk] + list(const_walk) + list(anc_walk))
    walk.x(const_walk[0])
    
    walk.x(const_walk[:])
    walk.x(ctrl_walk)
    walk.append(ctrl_adder, [ctrl_walk] + list(const_walk) + list(anc_walk))
    walk.x(ctrl_walk)
    walk.x(const_walk[:])
    

    walkGate = walk.to_gate()
    walkGate.label = 'walk'
    unwalkGate = walk.to_gate().inverse()
    unwalkGate.label = 'unwalk'

    #Executing each braid generator
    for gen in braid_sequence:
        
        #Walk gen-1 steps
        for i in range(abs(gen)-1):
                circ.append(walkGate, [path[i]] + list(anc) + list(const))
        
        if gen > 0:
            #Multiplexing: if ancilla is equal to v, then execute Phi_v
            for v in range(1, k):
                zero_bits = [i for i in range(anc_size) if not (v >> i) & 1] # For each vertex v, find the index of 0s in a binary expansion of v
                circ.x(anc[zero_bits]) # Flip all the qubits in zero_bits to 1. This way, the multi-controlled phiGates only activates when the ancilla is precisely v
                circ.append(ctrl_phiGates[v-1], list(anc) + [path[gen - 1], path[gen]]) # Execute Phi_v on the i-th and (i+1)-th qubit with ancilla as controls
                circ.x(anc[zero_bits]) # Undo the flip to prepare for the next v
        
        elif gen < 0:
            #Multiplexing: if ancilla is equal to v, then execute Phi_inv_v
            for v in range(1, k):
                zero_bits = [i for i in range(anc_size) if not (v >> i) & 1] # For each vertex v, find the index of 0s in a binary expansion of v
                circ.x(anc[zero_bits]) # Flip all the qubits in zero_bits to 1. This way, the multi-controlled phiGates only activates when the ancilla is precisely v
                circ.append(ctrl_phiGates_inv[v-1], list(anc) + [path[abs(gen) - 1], path[abs(gen)]]) # Execute Phi_v on the i-th and (i+1)-th qubit with ancilla as controls
                circ.x(anc[zero_bits]) # Undo the flip to prepare for the next v
        
        else:
            return print("Error: invalid braid generator")
        
        #Unwalk gen-1 steps
        for i in reversed(range(abs(gen)-1)):
                circ.append(unwalkGate, [path[i]] + list(anc) + list(const))
                
    return circ.to_gate(label='braid')
    

<>:9: SyntaxWarning: invalid escape sequence '\e'
<>:9: SyntaxWarning: invalid escape sequence '\e'
C:\Users\Edward\AppData\Local\Temp\ipykernel_41112\2181604927.py:9: SyntaxWarning: invalid escape sequence '\e'
  according to the distribution Pr(p) ∝ λ_{\ell(p)}.


In [20]:
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit.quantum_info import SparsePauliOp

def jonespoly(strand_count, braid, k):
    braidGate = create_braidGate(strand_count, braid, k)
    path_input = sample_path(strand_count, k)

    # Creating Hadamard test for the real component

    ctrl_braidGate = braidGate.control(1)

    had = QuantumRegister(1, 'had')
    path = QuantumRegister(strand_count,'path')
    anc = QuantumRegister(anc_size, 'anc')
    const = QuantumRegister(anc_size,'const')

    reHad_circ = QuantumCircuit(had, path, anc, const)
    reHad_circ.h(had[0])
    
    for idx, bit in enumerate(path_input):
        if bit == 1:
            reHad_circ.x(path[idx])

    reHad_circ.x(anc[0])
    reHad_circ.append(ctrl_braidGate, [had[0]] + list(path) + list(anc) + list(const))
    reHad_circ.h(had[0])

    # Creating Hadamard test for the imaginary component
    imHad_circ = QuantumCircuit(had, path, anc, const)
    imHad_circ.h(had[0])
    imHad_circ.sdg(had[0])

    for idx, bit in enumerate(path_input):
        if bit == 1:
            imHad_circ.x(path[idx])

    imHad_circ.x(anc[0])
    imHad_circ.append(ctrl_braidGate, [had[0]] + list(path) + list(anc) + list(const))
    imHad_circ.h(had[0])
    num_qubits = 1 + strand_count + 2*anc_size
    obs_str = 'I'*(num_qubits-1) + 'Z'
    obs = SparsePauliOp(obs_str, coeffs=1.0)

    estimator = Estimator()
    job = estimator.run([(reHad_circ, obs),(imHad_circ, obs)], precision=0.01)
    values = job.result()
    reInner, imInner = values[0], values[1]
    
    inner = complex(round(reInner.data.evs[()],5), round(imInner.data.evs[()],5))
    return inner

In [21]:
job = [2, [1,1], 5]
trial = 2**10
trial_log = []

#Pre compute:
k = job[2]
lam = [0] + [np.sin(np.pi*v/k) for v in range(1,k)] + [0]
A = (0+1j)*np.exp((0-1j)*np.pi/(2*k))
anc_size = int(np.ceil(np.log2(2*k)))
d = -A**2-A**(-2)

# Creating gates that execute the representation of braid generators
phi = [A*np.array([[0,0,0,0],
                    [0,lam[v+1]/lam[v],np.sqrt(lam[v+1]*lam[v-1])/lam[v],0],
                    [0,np.sqrt(lam[v+1]*lam[v-1])/lam[v],lam[v-1]/lam[v],0],
                    [0,0,0,0]]) + A**(-1)*np.identity(4) for v in range(1,k)]
phiGates = []
for i in range(len(phi)):
    phiCirc = QuantumCircuit(2)
    phiCirc.unitary(phi[i],[0,1])
    phiGates.append(phiCirc.to_gate(label=f"phi_{i}"))
ctrl_phiGates = [U.control(anc_size) for U in phiGates]

# Creating gates that execute the representation of inverse braid generators
phi_inv = [A**(-1)*np.array([[0,0,0,0],
                    [0,lam[v+1]/lam[v],np.sqrt(lam[v+1]*lam[v-1])/lam[v],0],
                    [0,np.sqrt(lam[v+1]*lam[v-1])/lam[v],lam[v-1]/lam[v],0],
                    [0,0,0,0]]) + A*np.identity(4) for v in range(1,k)]
phiGates_inv = []
for i in range(len(phi_inv)):
    phiCirc = QuantumCircuit(2)
    phiCirc.unitary(phi_inv[i],[0,1])
    phiGates_inv.append(phiCirc.to_gate(label=f"phi_inv_{i}"))
ctrl_phiGates_inv = [U.control(anc_size) for U in phiGates_inv]

#Computing trial-many paths and logging them 
for i in range(trial):
    trial_log += [jonespoly(job[0],job[1],job[2])]
inner = np.mean(trial_log)
writhe = sum(1 if g > 0 else -1 for g in job[1])    
jones = (-A)**(-3*writhe)*d**(job[0]-1)*inner

print(f"Braid {job[1]} in B{job[0]} inner product value at k={job[2]}: {jones}.")

Braid [1, 1] in B2 inner product value at k=5: (-0.23399980519447433+0.602179386000759j).
